# Experiment 7: t2 Dual-Role &epsilon; Test

Builds on the shared-tree PCFG infrastructure from Experiment 6/6b (base grammar,
`permute_grammar`, budget-matching, fixed-step "convergence-matched" training, A'
calibration, d-sweep) &mdash; **reused unchanged** wherever the design doc says so.

**What's different from Exp 6b:** Exp 6b's terminal-collapse *syncretism* mechanism
(`eps_level` collapse maps) is **not used here**. It was diagnosed and abandoned
(see project memory) as testing source-side information loss, not anchor-bias
&epsilon; &mdash; a dedicated/separate circuit trained on the same lossy input would
suffer the identical floor, so the mechanism never isolated anchor-bias in the first
place. This notebook implements its lossless successor instead: the **t2 dual-role**
mechanism. One terminal (`t2`) plays one of two grammatical roles (ROLE-X / ROLE-Y)
depending on which of two designated rules fired &mdash; always 100% recoverable from
tree structure, never destroyed, never ambiguous. The word "syncretism" does not
appear anywhere below except in this contrast note.

**A causal-ordering fix not spelled out in the design doc, added here:** the design
doc places `t_agree` "structurally distant" from `t2` inside the recursively-linearized
tree. But L_C's linearization is a **full reversal of every level** (confirmed in Exp
6/6b: `L_C == reverse(L_A)`), so *any* fixed placement of two leaves inside the tree
has their relative order flip for L_C &mdash; `t_agree` would end up *before* `t2` in
the token stream, making the task structurally unsolvable for L_C in a causal LM (not
an &epsilon; effect, just an impossible read-before-write). This is the exact same
class of bug the S5-relay design hit and fixed (see project memory: generators moved
to a fixed preamble immune to reversal). The mirror-image fix is used here: `t_agree`
is appended as a **fixed postamble token, after the entire linearized tree**, which is
immune to any per-level reversal by construction and guarantees `t2` always precedes
`t_agree` in every language's rendering. This is flagged explicitly for review.


## 0. Device setup (reused, unchanged)

In [1]:
import os
import torch
DEVICE = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  {torch.cuda.get_device_name(2)}")

from contextlib import nullcontext
def autocast_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16) if DEVICE.type == "cuda" else nullcontext()


using device: cuda:4
  NVIDIA H200 NVL


## 1. Config

Base PCFG constants (`LEVELS`, `NT_SIZE`, `TERM_SIZE`, `MIN/MAX_DEGREE`,
`RULE_LEN_CHOICES`, `LANGS`, `MIX`, `BOS`) are reused verbatim from Exp 6/6b.
New: the dual-role config (`T2`, `NT_X`/`NT_Y` role weights, `AGREE_X_ID`/`AGREE_Y_ID`
postamble vocabulary, reweighting factors).

In [2]:
# ============================== CONFIG (base, reused) ========================
LEVELS = 6            # number of NT layers (root=level 1 ... level 6); terminals at level 7
NT_SIZE = 4            # symbols per NT layer, levels 2..6 (root layer has size 1)
TERM_SIZE = 4          # number of terminal symbols
MIN_DEGREE = 2         # min number of production rules per NT symbol
MAX_DEGREE = 4         # max number of production rules per NT symbol
RULE_LEN_CHOICES = [2, 3]   # rule RHS lengths

LANGS = {
    "L_A": set(),                          # identity
    "L_B": {4, 5},                         # partial reversal (irregular)
    "L_C": set(range(1, LEVELS + 1)),      # full reversal (regular, = reverse(L_A))
}
LANG_NAMES = list(LANGS.keys())
MIX = {"L_A": 0.70, "L_B": 0.15, "L_C": 0.15}   # default training-data mix, anchor dominates -- ρ-sweep (Step 4) varies this

BOS_OFFSET = TERM_SIZE
BOS = {name: BOS_OFFSET + i for i, name in enumerate(LANG_NAMES)}
VOCAB_SIZE_BASE = TERM_SIZE + len(LANG_NAMES)     # terminals + BOS tags, no agree tokens yet

def round_to_valid_n_embd(x, n_head):
    """Rotary embeddings need head_dim = n_embd/n_head to be EVEN."""
    unit = 2 * n_head
    return max(unit, round(x / unit) * unit)

# --------------------------- t2 DUAL-ROLE (new) -------------------------------
# t2: the designated terminal that plays one of two roles depending on which
# rule fired. NT_X / NT_Y are two level-LEVELS symbols with a SINGLE,
# IDENTICAL rule each (both produce t2 deterministically); which one gets
# visited is controlled by ROLE_CHOICE, one level up, via a per-language
# weighted 2-way rule choice (see build_dual_role_grammar/rule_weights_for
# in Section 2 for the full position-controlled trunk this sits on, and why).
T2 = TERM_SIZE - 1

W_HI, W_LO = 9.0, 1.0     # ROLE_CHOICE's ~90/10 split, per language -- see rule_weights_for

REWEIGHT_FACTORS = [1, 5, 20, 100]   # Step 2 sweep
RHO_SWEEP = [0.50, 0.70, 0.85, 0.95] # Step 4 sweep (anchor's mixture share)

print(f"t2 = terminal {T2} (of {TERM_SIZE} terminals); dual-role weights W_HI={W_HI}, W_LO={W_LO}")


t2 = terminal 3 (of 4 terminals); dual-role weights W_HI=9.0, W_LO=1.0


## 2. Grammar builder, dual-role sampler, marked linearizer

`build_grammar`, `sample_tree`, `linearize`, `permute_grammar` below are reused
**unchanged** from Exp 6/6b (kept for parity / used elsewhere, e.g. `sample_tree`
by other tooling). Everything under "dual-role additions" is new: it does not
modify the reused functions, only adds to the same rules dict they operate on.

In [3]:
import random
from collections import defaultdict

def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))

    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set()
            tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen:
                    continue
                seen.add(rhs)
                rules[sym].append(rhs)
    return rules, symbols_by_level


def sample_tree(rules, rng, symbol=(1, 0), level=1):
    """Plain (uniform, non-indexed) sampler -- reused unchanged, kept for
    parity/other tooling. NOT used for dual-role data (see sample_dual_role_tree)."""
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)


def linearize(node, reverse_levels):
    """True-terminal linearization, word order only -- reused unchanged."""
    if isinstance(node, int):
        return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize(c, reverse_levels))
    return tokens


def permute_grammar(rules, symbols_by_level, seed):
    """Disjoint-control grammar via random rule-ownership shuffle -- reused
    unchanged from Exp 6b, kept for parity / available if a disjoint-abstract-
    grammar control is wanted later. NOT used to derive L_B/L_C's dual-role
    weights below (see config cell note: a full random shuffle wouldn't
    reliably swap exactly NT_X<->NT_Y, so that swap is done directly)."""
    rng = random.Random(seed)
    new_rules = {(1, 0): rules[(1, 0)]}
    for level, syms in symbols_by_level.items():
        if level == 1:
            continue
        shuffled = syms[:]
        rng.shuffle(shuffled)
        for original_sym, new_owner in zip(syms, shuffled):
            new_rules[new_owner] = rules[original_sym]
    return new_rules


# ------------------------- dual-role additions (new) --------------------------

def build_dual_role_grammar(seed):
    """Reuses build_grammar (unchanged) for the base PCFG, then restructures a
    single controlled path so t2 sits at the structural MIDDLE of the tree:

      ROOT (level 1) is overridden to a FIXED 3-way split
      (FILLER_L, TRUNK2, FILLER_R). FILLER_L/FILLER_R are ordinary,
      unmodified (except t2-scrubbed) recursive PCFG subtrees -- they supply
      realistic bulk content on BOTH sides of t2. TRUNK2 heads a single-child
      pass-through chain (TRUNK2 -> TRUNK3 -> TRUNK4 -> ROLE_CHOICE) straight
      down to level 5, where ROLE_CHOICE is the ONE weighted decision point:
      rules[ROLE_CHOICE] = [(NT_X,), (NT_Y,)]. NT_X/NT_Y (level 6) each have a
      SINGLE deterministic rule, byte-for-byte identical: (other_x, T2, other_y).

      Every other rule in the grammar is scrubbed of accidental references to
      {TRUNK2, TRUNK3, TRUNK4, ROLE_CHOICE, NT_X, NT_Y}, so this trunk is the
      ONLY path to t2 -- exactly one occurrence per tree, always.

    WHY: an earlier version let the ordinary random grammar decide how many
    times and where NT_X/NT_Y got visited. Empirically (checked across
    several seeds) NT_X/NT_Y got visited MANY times per tree, and "the first
    hit among many independent trials" is a basic probability effect that
    lands very early in canonical (L_A) order (median relative position
    ~2-5%). Under L_C's FULL reversal that flips to very LATE -- i.e. right
    next to the t_agree postamble (measured median distance ~7 tokens, vs.
    ~240 for L_A/L_B), silently turning L_C's task into a near-local lookup
    instead of the intended long-distance one. A 3-way split's MIDDLE child
    stays the middle child under any reversal (reversing an odd-length list
    fixes its center), so t2's distance to BOTH the BOS and the postamble
    stays comparable to FILLER_L/FILLER_R's length for every language,
    regardless of reversal. Verified below and across seeds.
    """
    rules, symbols_by_level = build_grammar(seed)
    lvl2, lvl3, lvl4, lvl5, lvl6 = (symbols_by_level[l] for l in (2, 3, 4, 5, 6))
    trunk2, filler_l, filler_r = lvl2[0], lvl2[1], lvl2[2]
    trunk3 = lvl3[0]
    trunk4 = lvl4[0]
    role_choice = lvl5[0]
    nt_x, nt_y = lvl6[0], lvl6[1]

    reserved_by_level = {2: {trunk2}, 3: {trunk3}, 4: {trunk4}, 5: {role_choice}, 6: {nt_x, nt_y}}

    # scrub every OTHER symbol's references to a reserved child, level by
    # level, so each reserved symbol is reachable via EXACTLY the fixed trunk.
    for level in range(1, LEVELS):
        child_level = level + 1
        reserved = reserved_by_level[child_level]
        safe_children = [s for s in symbols_by_level[child_level] if s not in reserved]
        fallback = safe_children[0]
        for sym in symbols_by_level[level]:
            if sym in reserved_by_level.get(level, set()):
                continue   # this symbol's own rules get overridden below anyway
            rules[sym] = [tuple(fallback if c in reserved else c for c in rhs) for rhs in rules[sym]]

    # scrub t2 out of every level-6 symbol EXCEPT nt_x/nt_y (unchanged from before)
    fallback_term = 0 if T2 != 0 else 1
    for sym in lvl6:
        if sym in (nt_x, nt_y):
            continue
        rules[sym] = [tuple(fallback_term if t == T2 else t for t in rhs) for rhs in rules[sym]]

    # fixed trunk overrides
    rules[(1, 0)] = [(filler_l, trunk2, filler_r)]      # ROOT: t2's ancestor is the MIDDLE child
    rules[trunk2] = [(trunk3,)]
    rules[trunk3] = [(trunk4,)]
    rules[trunk4] = [(role_choice,)]
    rules[role_choice] = [(nt_x,), (nt_y,)]             # THE weighted choice point
    other_x = (T2 + 1) % TERM_SIZE
    other_y = (T2 + 2) % TERM_SIZE
    rules[nt_x] = [(other_x, T2, other_y)]              # single, deterministic
    rules[nt_y] = [(other_x, T2, other_y)]              # IDENTICAL to nt_x -- no local tell either

    return rules, symbols_by_level, role_choice, nt_x, nt_y


def rule_weights_for(lang, role_choice):
    """L_A: ~90% ROLE_CHOICE picks NT_X (ROLE-X-favoring); L_B/L_C: ~90% picks
    NT_Y instead (ROLE-Y-favoring) -- the targeted swap described in the
    earlier config-cell note, now applied at the single ROLE_CHOICE decision
    point rather than duplicated across NT_X/NT_Y. L_B and L_C share the SAME
    weights (both are "the non-anchor languages"); they differ only in
    linearization order (LANGS), exactly as in Exp 6/6b."""
    if lang == "L_A":
        return {role_choice: [W_HI, W_LO]}
    return {role_choice: [W_LO, W_HI]}


def sample_tree_dualrole(rules, weights, nt_x, nt_y, rng):
    """Weighted PCFG sampler: identical recursive shape to the reused
    sample_tree, except symbols present in `weights` (i.e. now just
    ROLE_CHOICE) use rng.choices with those weights instead of rng.choice's
    implicit uniform draw; every other symbol samples uniformly, unchanged.
    Also marks the (now unique, guaranteed) t2-rule firing at NT_X/NT_Y --
    this is the tree's fixed, language-invariant 'role' ground truth, decided
    once here, before any per-language linearization/reversal. Returns
    (tree, role) with role in {'X','Y',None}; None should not occur given the
    position-controlled trunk in build_dual_role_grammar (kept only as a
    defensive signal in case that invariant is ever broken elsewhere)."""
    state = {"role": None, "marked": False}

    def rec(symbol, level):
        opts = rules[symbol]
        if symbol in weights:
            i = rng.choices(range(len(opts)), weights=weights[symbol])[0]
        else:
            i = rng.randrange(len(opts))
        rhs = opts[i]
        children = []
        for child_sym in rhs:
            if level == LEVELS:
                if symbol in (nt_x, nt_y) and i == 0 and child_sym == T2 and not state["marked"]:
                    state["marked"] = True
                    state["role"] = "X" if symbol == nt_x else "Y"
                    children.append(("T2MARK", child_sym))
                else:
                    children.append(child_sym)
            else:
                children.append(rec(child_sym, level + 1))
        return (symbol, children)

    tree = rec((1, 0), 1)
    return tree, state["role"]


def sample_dual_role_tree(rules, weights, nt_x, nt_y, rng, max_tries=200):
    """Rejection-samples until a tree with a defined role is produced. With
    the position-controlled trunk (Section 2), every tree deterministically
    visits the trunk and fires a role exactly once, so this should always
    succeed on the first try -- the loop is kept as a defensive fallback,
    not because rejection is expected in normal operation."""
    for _ in range(max_tries):
        tree, role = sample_tree_dualrole(rules, weights, nt_x, nt_y, rng)
        if role is not None:
            return tree, role
    raise RuntimeError("no NT_X/NT_Y t2-rule firing after max_tries -- check grammar/weights")


def linearize_marked(node, reverse_levels):
    """Like linearize(), but tracks the T2MARK leaf's position through
    whatever per-level reversal this language applies. Returns (tokens,
    marked_pos); marked_pos is None if no marked leaf is in this subtree."""
    if isinstance(node, tuple) and len(node) == 2 and node[0] == "T2MARK":
        return [node[1]], 0
    if isinstance(node, int):
        return [node], None
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens, mpos = [], None
    for c in order:
        t, p = linearize_marked(c, reverse_levels)
        if p is not None:
            mpos = len(tokens) + p
        tokens.extend(t)
    return tokens, mpos


# smoke test
_rules0, _syms0, _ROLE0, _NTX0, _NTY0 = build_dual_role_grammar(seed=0)
_rng0 = random.Random(1)
_wA = rule_weights_for("L_A", _ROLE0)
_wB = rule_weights_for("L_B", _ROLE0)
_roles_A = [sample_dual_role_tree(_rules0, _wA, _NTX0, _NTY0, _rng0)[1] for _ in range(2000)]
_roles_B = [sample_dual_role_tree(_rules0, _wB, _NTX0, _NTY0, _rng0)[1] for _ in range(2000)]
print(f"L_A empirical P(ROLE-X) = {_roles_A.count('X')/len(_roles_A):.3f}  (target ~0.9)")
print(f"L_B empirical P(ROLE-X) = {_roles_B.count('X')/len(_roles_B):.3f}  (target ~0.1, i.e. ROLE-Y-favoring)")

# distance check: t2's distance to the postamble should now be COMPARABLE
# across languages, not near-zero for L_C -- this is exactly the bug that
# an earlier version of this grammar had (see build_dual_role_grammar's
# docstring) and this smoke test exists specifically to catch a regression.
import statistics as _stats
for _lang, _w in (("L_A", _wA), ("L_B", _wB), ("L_C", _wB)):
    _rng_d = random.Random(9)
    _dists = []
    for _ in range(1500):
        _tree, _role = sample_dual_role_tree(_rules0, _w, _NTX0, _NTY0, _rng_d)
        _toks, _mpos = linearize_marked(_tree, LANGS[_lang])
        _dists.append(len(_toks) - _mpos)
    print(f"{_lang}: median dist(t2 -> postamble) = {_stats.median(_dists):.0f} "
          f"(should be comparable across all three languages, NOT near-zero for L_C)")


L_A empirical P(ROLE-X) = 0.903  (target ~0.9)
L_B empirical P(ROLE-X) = 0.096  (target ~0.1, i.e. ROLE-Y-favoring)
L_A: median dist(t2 -> postamble) = 65 (should be comparable across all three languages, NOT near-zero for L_C)
L_B: median dist(t2 -> postamble) = 65 (should be comparable across all three languages, NOT near-zero for L_C)
L_C: median dist(t2 -> postamble) = 79 (should be comparable across all three languages, NOT near-zero for L_C)


## 3. Model &mdash; TinyRotaryGPT (unchanged from Exp 6/6b)

In [4]:
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def forward_with_patch(self, idx, patch_layer=None, patch_mask=None, patch_values=None):
        """Manual forward that can overwrite the residual stream right after
        block `patch_layer`, at positions where patch_mask is True, with
        patch_values. Used by the interchange intervention (Section 12)."""
        x = self.tok_emb(idx)
        for l, blk in enumerate(self.blocks):
            x = blk(x)
            if patch_layer is not None and l == patch_layer:
                x = x.clone()
                x[patch_mask] = patch_values.to(x.dtype)
        return x

# smoke test
torch.manual_seed(0)
_m = TinyRotaryGPT(VOCAB_SIZE_BASE + 2, n_embd=8, n_layer=2, n_head=2, max_seq_len=64).to(DEVICE)
_idx = torch.randint(0, VOCAB_SIZE_BASE + 2, (4, 32), device=DEVICE)
_logits, _ = _m(_idx, _idx)
print(f"forward pass OK on {DEVICE}, output shape:", _logits.shape)


forward pass OK on cuda:4, output shape: torch.Size([4, 32, 9])


## 4. Grammar instance, AGREE vocabulary, and run constants

`AGREE_X_ID`/`AGREE_Y_ID` are two NEW vocabulary slots (beyond the base terminals +
BOS tags) reserved exclusively for the `t_agree` postamble token, so it can never be
confused with an ordinary grammar terminal.

In [5]:
AGREE_X_ID = VOCAB_SIZE_BASE
AGREE_Y_ID = VOCAB_SIZE_BASE + 1
VOCAB_SIZE = VOCAB_SIZE_BASE + 2     # base vocab (terminals + BOS) + 2 agree tokens

def agree_token_for(role):
    return AGREE_X_ID if role == "X" else AGREE_Y_ID

rules, symbols_by_level, ROLE_CHOICE, NT_X, NT_Y = build_dual_role_grammar(seed=0)
RULE_WEIGHTS = {lang: rule_weights_for(lang, ROLE_CHOICE) for lang in LANG_NAMES}

N_TREES_CHECK = 50_000
SEQ_LEN_CHECK = 256
FIXED_BUDGET_SWEEP = 90_000     # "convergence-matched" via fixed generous step budget,
                                 # per the abandoned-adaptive-convergence lesson (Exp 6/6b)
N_HEAD = 4
SEEDS3 = [0, 7, 13]
D_LIST = [8, 16, 32, 96]         # reused default capacity grid
MAX_PAR = 20                     # parallel training workers

for d in D_LIST:
    assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0, f"d={d}: head_dim must be even for rotary"

print(f"grammar built (seed=0): NT_X={NT_X}, NT_Y={NT_Y}, T2={T2}, "
      f"AGREE_X_ID={AGREE_X_ID}, AGREE_Y_ID={AGREE_Y_ID}, VOCAB_SIZE={VOCAB_SIZE}")


grammar built (seed=0): NT_X=(6, 0), NT_Y=(6, 1), T2=3, AGREE_X_ID=7, AGREE_Y_ID=8, VOCAB_SIZE=9


## 5. Pre-flight checks (NO training) &mdash; Section 3 of the design doc

Must all pass cleanly before any training (Section 3's own instruction, echoing the
Exp 6b lesson: syncretism's design was found broken only *after* training because its
pre-flight check's headline number was misleading).

**3.1 shortcut closure.** Can `role(t2)` be predicted above chance from context other
than `t2`'s own token (sibling terminals / local n-gram window), for L_B and L_C, in
their own rendering? Window swept well past 32 tokens (Exp 6b's syncretism pre-flight
found this grammar's low branching factor lets nearby siblings leak information at
surprisingly long range).

**3.1 reverse-direction bullet.** In this design there is no separate observable
"rule-choice node" apart from `t2`'s own token: whether the t2-rule fired at NT_X/NT_Y
is only ever visible through whether `t2` appears there at all. So the reverse-leak
check collapses into the forward check above by construction &mdash; noted rather than
implemented separately.

**3.2 balance check.** Is `t_agree`'s label distribution close to 50/50?

**3.3 repetition/ambiguity audit.** Restrict 3.1's check to contexts seen &ge;2 times
in the sample (Exp 6b's over-aggressive-syncretism lesson: a check averaged over
mostly-unique contexts is misleadingly optimistic) &mdash; reported as a separate
column in the same table, not a separate function.

In [6]:
def preflight_shortcut_closure(rules, weights, lang, n_trees=20_000,
                               window_sizes=(4, 8, 16, 32, 64, 128), seed=0):
    rng = random.Random(seed)
    ctx_to_role = {K: {} for K in window_sizes}
    n_used = 0
    for _ in range(n_trees):
        tree, role = sample_dual_role_tree(rules, weights, NT_X, NT_Y, rng)
        toks, mpos = linearize_marked(tree, LANGS[lang])
        n_used += 1
        for K in window_sizes:
            ctx = tuple(toks[max(0, mpos - K):mpos])
            ctx_to_role[K].setdefault(ctx, []).append(role)

    print(f"\n[{lang}] pre-flight shortcut-closure check over {n_used} trees")
    print(f"{'window K':>9s} {'unique ctx':>11s} {'ambig (all ctx)':>16s} "
          f"{'unique ctx seen>=2':>19s} {'ambig (seen>=2)':>16s}")
    out = {}
    for K in window_sizes:
        d = ctx_to_role[K]
        n_ambig_all = sum(1 for v in d.values() if len(set(v)) > 1)
        rep = {c: v for c, v in d.items() if len(v) >= 2}
        n_ambig_rep = sum(1 for v in rep.values() if len(set(v)) > 1)
        frac_rep = n_ambig_rep / max(1, len(rep))
        out[K] = dict(n_contexts=len(d), n_ambig_all=n_ambig_all,
                      n_repeated_contexts=len(rep), frac_ambig_repeated=frac_rep)
        print(f"{K:>9d} {len(d):>11d} {n_ambig_all:>16d} {len(rep):>19d} {frac_rep:>16.3f}")
    return out


def preflight_balance_check(rules, weights, lang, n_trees=20_000, seed=1):
    rng = random.Random(seed)
    roles = [sample_dual_role_tree(rules, weights, NT_X, NT_Y, rng)[1] for _ in range(n_trees)]
    frac_x = roles.count("X") / len(roles)
    print(f"[{lang}] t_agree label balance over {len(roles)} trees: "
          f"ROLE-X={frac_x:.3f}  ROLE-Y={1 - frac_x:.3f}")
    return frac_x


print("=" * 78)
print("3.1/3.3: if 'ambig (seen>=2)' stays > 0 well past K=32, role(t2) requires")
print("genuine long-range tracking, not a local shortcut. If it hits 0 quickly,")
print("consider this a closed instrument (no leak) rather than a design failure.")
print("CAUTION: once 'unique ctx seen>=2' itself drops toward 0 at large K, the")
print("ambig-fraction is computed over an empty/tiny repeated-context pool and")
print("reads as 0.000 with NO real evidence behind it -- this is the exact")
print("Exp 6b false-confidence trap in a different shape. Trust a 0.000 only at")
print("a K where 'unique ctx seen>=2' is still a substantial count.")
print("=" * 78)
for lang in LANG_NAMES:
    preflight_shortcut_closure(rules, RULE_WEIGHTS[lang], lang)

print("\n" + "=" * 78)
print("3.2: balance check -- should be close to 50/50 for each language.")
print("=" * 78)
for lang in LANG_NAMES:
    preflight_balance_check(rules, RULE_WEIGHTS[lang], lang)

print("\nDo not proceed past this cell until both checks look acceptable.")


3.1/3.3: if 'ambig (seen>=2)' stays > 0 well past K=32, role(t2) requires
genuine long-range tracking, not a local shortcut. If it hits 0 quickly,
consider this a closed instrument (no leak) rather than a design failure.
CAUTION: once 'unique ctx seen>=2' itself drops toward 0 at large K, the
ambig-fraction is computed over an empty/tiny repeated-context pool and
reads as 0.000 with NO real evidence behind it -- this is the exact
Exp 6b false-confidence trap in a different shape. Trust a 0.000 only at
a K where 'unique ctx seen>=2' is still a substantial count.

[L_A] pre-flight shortcut-closure check over 20000 trees
 window K  unique ctx  ambig (all ctx)  unique ctx seen>=2  ambig (seen>=2)
        4          14               14                  14            1.000
        8         295              269                 293            0.918
       16       16925              513                2481            0.207
       32       20000                0                   0            

## 6. Data generation (dual-role, with t_agree postamble)

Each language draws trees using **its own** `RULE_WEIGHTS` &mdash; this is what
creates the anchor bias (L_A's weights favor ROLE-X at NT_X; L_B/L_C's favor
ROLE-Y). `t_agree`'s label is `agree_token_for(role)`, appended as a fixed
postamble token immediately after the full tree (see the notebook-header note on
why this must be a postamble, not spliced inside the tree).

`make_dual_role_dedicated_matched` implements Section 5 of the design doc: the
dedicated baseline must see exactly `mix[lang] * n_total_trees` native trees (not
lang's full monolingual stream), repeated/tiled to the shared model's total chunk
count, so "no sharing cost" isn't confounded with "saw more data".

In [7]:
def make_dual_role_mix(rules, n_trees, seed, seq_len, mix=MIX):
    rng = random.Random(seed)
    stream = []
    for _ in range(n_trees):
        lang = rng.choices(LANG_NAMES, weights=[mix[n] for n in LANG_NAMES])[0]
        tree, role = sample_dual_role_tree(rules, RULE_WEIGHTS[lang], NT_X, NT_Y, rng)
        toks, _ = linearize_marked(tree, LANGS[lang])
        stream.append(BOS[lang])
        stream.extend(toks)
        stream.append(agree_token_for(role))
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


def make_dual_role_monolingual(rules, lang_name, n_trees, seed, seq_len):
    rng = random.Random(seed)
    stream = []
    for _ in range(n_trees):
        tree, role = sample_dual_role_tree(rules, RULE_WEIGHTS[lang_name], NT_X, NT_Y, rng)
        toks, _ = linearize_marked(tree, LANGS[lang_name])
        stream.append(BOS[lang_name])
        stream.extend(toks)
        stream.append(agree_token_for(role))
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


def make_dual_role_dedicated_matched(rules, lang_name, seed, seq_len, target_chunks, mix=MIX,
                                     n_total_trees=N_TREES_CHECK):
    """Data-volume-matched dedicated baseline (Section 5): dedicated-B sees
    exactly mix[lang]*n_total_trees NATIVE B trees, tiled up to target_chunks
    (the shared model's own chunk count) so both conditions train on the same
    total token budget."""
    n_native = max(1, int(round(mix[lang_name] * n_total_trees)))
    native = make_dual_role_monolingual(rules, lang_name, n_native, seed, seq_len)
    reps = max(1, -(-target_chunks // native.shape[0]))
    print(f"  [{lang_name}] native trees={n_native}, native chunks={native.shape[0]}, "
          f"tiled {reps}x -> {min(target_chunks, native.shape[0]*reps)} chunks (repetition ratio {reps}x)")
    tiled = native.repeat(reps, 1)[:target_chunks]
    return tiled


def build_matched_probe_set(rules, n_trees, seed, seq_len):
    """Fixed set of trees (role drawn once via L_A's weights) each rendered
    under all three languages' linearizations, for the Step 5 representational
    probe / interchange intervention -- reused pattern from Exp6b's
    build_matched_eval_set_eps (same tree, different renderings)."""
    rng = random.Random(seed)
    out = {lang: dict(idx=[], t2_pos=[], role=[]) for lang in LANG_NAMES}
    kept = tries = 0
    while kept < n_trees and tries < n_trees * 30:
        tries += 1
        tree, role = sample_dual_role_tree(rules, RULE_WEIGHTS["L_A"], NT_X, NT_Y, rng)
        rows, ok = {}, True
        for lang in LANG_NAMES:
            toks, mpos = linearize_marked(tree, LANGS[lang])
            if mpos is None or len(toks) + 2 > seq_len:
                ok = False
                break
            rows[lang] = (toks, mpos)
        if not ok:
            continue
        kept += 1
        for lang in LANG_NAMES:
            toks, mpos = rows[lang]
            row = [BOS[lang]] + toks + [agree_token_for(role)]
            row = row + [0] * (seq_len - len(row))
            out[lang]["idx"].append(row)
            out[lang]["t2_pos"].append(mpos + 1)   # +1 for BOS offset
            out[lang]["role"].append(0 if role == "X" else 1)
    for lang in LANG_NAMES:
        out[lang]["idx"] = torch.tensor(out[lang]["idx"], dtype=torch.long)
        out[lang]["t2_pos"] = torch.tensor(out[lang]["t2_pos"], dtype=torch.long)
        out[lang]["role"] = torch.tensor(out[lang]["role"], dtype=torch.long)
    return out


# smoke test
_d = make_dual_role_mix(rules, n_trees=200, seed=1, seq_len=SEQ_LEN_CHECK)
_n_agree = int(((_d == AGREE_X_ID) | (_d == AGREE_Y_ID)).sum())
print(f"smoke: {_d.shape[0]} chunks, {_n_agree} agree-token positions in sample")
_ps = build_matched_probe_set(rules, n_trees=20, seed=2, seq_len=SEQ_LEN_CHECK)
print(f"matched probe set: {_ps['L_A']['idx'].shape[0]} trees kept, "
      f"L_A t2_pos sample={_ps['L_A']['t2_pos'][:5].tolist()}, "
      f"L_C t2_pos sample={_ps['L_C']['t2_pos'][:5].tolist()}")


smoke: 116 chunks, 198 agree-token positions in sample
matched probe set: 20 trees kept, L_A t2_pos sample=[78, 62, 68, 73, 87], L_C t2_pos sample=[66, 44, 66, 47, 61]


## 7. Budget-matched dedicated width calculator (reused, unchanged)

In [8]:
def count_params(d, n_layer=12, n_head=N_HEAD, seq_len=SEQ_LEN_CHECK, vocab_size=VOCAB_SIZE):
    m = TinyRotaryGPT(vocab_size, n_embd=d, n_layer=n_layer, n_head=n_head, max_seq_len=seq_len + 1)
    return sum(p.numel() for p in m.parameters())


def find_budget_matched_width(d_shared, K=3, n_head=N_HEAD, candidates=None):
    """Smallest-error d' (rotary-valid) such that K * params(d') ~= params(d_shared)."""
    target = count_params(d_shared, n_head=n_head)
    if candidates is None:
        unit = 2 * n_head
        candidates = list(range(unit, d_shared + 1, unit))
    best = min(candidates, key=lambda dp: abs(K * count_params(dp, n_head=n_head) - target))
    ratio = K * count_params(best, n_head=n_head) / target
    return best, ratio


BM_WIDTHS = {}
print(f"{'d_shared':>9s} {'params(d)':>12s} {'d_prime':>8s} {'K*params(d_prime)/params(d)':>28s}")
for d in D_LIST:
    dp, ratio = find_budget_matched_width(d, K=3, n_head=N_HEAD)
    BM_WIDTHS[d] = dp
    print(f"{d:>9d} {count_params(d):>12,d} {dp:>8d} {ratio:>28.3f}")


 d_shared    params(d)  d_prime  K*params(d_prime)/params(d)
        8       10,624        8                        3.000
       16       39,680        8                        0.803
       32      153,088       16                        0.778
       96    1,344,000       56                        1.030


## 8. Training infrastructure (reused pattern; `train_worker.py` extended with
optional loss reweighting for Step 2)

`--reweight W` upweights per-token loss at positions where the target is
`AGREE_Y_ID` (the minority-for-the-shared-model outcome, per the design doc's
Step 2). `W=1` (the default) is the plain, unweighted baseline used everywhere
except the Step 2 sweep.

In [9]:
%%writefile train_worker.py
import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", type=int, required=True)
    ap.add_argument("--seq_len", type=int, required=True)
    ap.add_argument("--steps", type=int, default=90000)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--reweight", type=float, default=1.0)
    ap.add_argument("--agree_y_id", type=int, default=-1)   # -1 = reweighting disabled
    a = ap.parse_args()

    torch.set_num_threads(1)
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    data = torch.load(a.data).to(device)
    model = TinyRotaryGPT(a.vocab, n_embd=a.d, n_layer=a.n_layer, n_head=a.n_head,
                          max_seq_len=a.seq_len + 1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, fused=True)
    n = data.shape[0]
    buf = torch.zeros(a.steps, device=device)
    t0 = time.time()
    for step in range(a.steps):
        idx = torch.randint(0, n, (a.batch,), device=device)
        batch = data[idx]
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)
            ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), reduction="none")
            if a.reweight != 1.0 and a.agree_y_id >= 0:
                w = torch.where(y.reshape(-1) == a.agree_y_id,
                                torch.tensor(a.reweight, device=device), torch.tensor(1.0, device=device))
                loss = (ce * w).sum() / w.sum()
            else:
                loss = ce.mean()
        opt.zero_grad(); loss.backward(); opt.step()
        buf[step] = loss.detach().float()
        if (step + 1) % 5000 == 0:
            print(f"step {step+1:>7d}: mean loss over last 5000 = {buf[step-4999:step+1].mean().item():.4f}"
                  f"  ({(step + 1) / (time.time() - t0):.0f} it/s)", flush=True)
        if (step + 1) % 10000 == 0 or step == a.steps - 1:
            tmp = a.ckpt + ".tmp"
            torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                        "step": step + 1, "losses": buf[:step + 1].tolist()}, tmp)
            os.replace(tmp, a.ckpt)

if __name__ == "__main__":
    main()


Overwriting train_worker.py


In [10]:
import subprocess, sys, time

def ckpt_step(path):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location="cpu", weights_only=False)["step"]


def run_parallel_jobs(jobs, vocab_size, seq_len, max_par=MAX_PAR, device=DEVICE):
    """jobs: list of dicts with keys d, seed, data_path, ckpt_path, n_head, steps,
    and optionally reweight/agree_y_id. Skips jobs already complete. Reuses
    train_worker.py unchanged in shape -- only the DATA and the optional
    reweight flag differ across conditions in this notebook."""
    jobs = [j for j in jobs if ckpt_step(j["ckpt_path"]) != j["steps"]]
    print(f"{len(jobs)} runs to do, {max_par} at a time")

    def launch(j):
        log = open(j["ckpt_path"] + ".log", "w")
        cmd = [sys.executable, "train_worker.py", "--d", str(j["d"]), "--seed", str(j["seed"]),
               "--data", j["data_path"], "--ckpt", j["ckpt_path"], "--vocab", str(vocab_size),
               "--seq_len", str(seq_len), "--steps", str(j["steps"]), "--n_head", str(j["n_head"]),
               "--device", str(device),
               "--reweight", str(j.get("reweight", 1.0)), "--agree_y_id", str(j.get("agree_y_id", -1))]
        return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), j, log

    running = []
    while jobs or running:
        still = []
        for p, j, log in running:
            rc = p.poll()
            if rc is None:
                still.append((p, j, log))
            else:
                log.close()
                tag = f"d={j['d']} seed={j['seed']} ({os.path.basename(j['ckpt_path'])})"
                print(f"finished {tag}  exit code {rc}" + ("" if rc == 0 else "   <-- check the .log file"))
        running = still
        while jobs and len(running) < max_par:
            running.append(launch(jobs.pop(0)))
        time.sleep(10)
    print("all queued jobs finished (check exit codes above for failures)")


def assert_all_complete(paths_and_steps):
    missing = [(p, s, ckpt_step(p)) for p, s in paths_and_steps if ckpt_step(p) != s]
    assert not missing, f"incomplete runs (path, target_step, actual_step): {missing}"
    print(f"all {len(paths_and_steps)} runs complete")


## 9. Evaluation infrastructure (mostly reused; `t_agree_accuracy` is new)

The measured quantity per the design doc: **accuracy specifically at the
`t_agree` token position** &mdash; restricted-vocab (2-way: AGREE_X_ID vs
AGREE_Y_ID), per the project's established lesson that full-vocab argmax at a
restricted-choice position understates real feature-level accuracy.

**Split by true class, not aggregate.** Each language's own role distribution
is skewed ~90/10 by construction, so a model that ignores `t2` entirely and
always predicts the majority role for its BOS tag already scores ~90% in
aggregate. `t_agree_accuracy_split` below reports minority-class accuracy as
the primary number (the Exp 3 number-agreement "aggregate accuracy conflating
the easy 92% and hard 8% case" lesson, reapplied here) and the naive
majority-vote baseline for direct comparison.

In [11]:
def load_shared_ckpt(path, d_model, seq_len, vocab_size=VOCAB_SIZE, n_layer=12, n_head=N_HEAD, device=DEVICE):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = TinyRotaryGPT(vocab_size, n_embd=d_model, n_layer=n_layer, n_head=n_head,
                      max_seq_len=seq_len + 1).to(device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    tail = ckpt["losses"][-1000:]
    print(f"{path}: step={ckpt['step']}, tail-1000 mean loss={sum(tail)/len(tail):.4f}")
    return m


@torch.no_grad()
def extract_hidden_states(model, idx_tensor, layers, batch_size=32, device=DEVICE):
    model.eval()
    activations = {l: [] for l in layers}

    def make_hook(l):
        def hook(module, inp, out):
            activations[l].append(out.detach().float().cpu())
        return hook

    handles = [model.blocks[l].register_forward_hook(make_hook(l)) for l in layers]
    try:
        for i in range(0, idx_tensor.shape[0], batch_size):
            batch = idx_tensor[i:i + batch_size].to(device)
            with autocast_ctx():
                _ = model(batch)
    finally:
        for h in handles:
            h.remove()
    return {l: torch.cat(activations[l], dim=0) for l in layers}


def split_trees(n_trees, test_frac=0.2, seed=0):
    import numpy as np
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n_trees)
    n_test = int(n_trees * test_frac)
    return set(idx[n_test:].tolist()), set(idx[:n_test].tolist())


@torch.no_grad()
def mean_ce(model, data, bs=256, device=DEVICE):
    model.eval(); tot = n = 0
    for i in range(0, data.shape[0], bs):
        b = data[i:i + bs].to(device)
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        tot += loss.item() * x.numel(); n += x.numel()
    return tot / n


@torch.no_grad()
def t_agree_accuracy_split(model, data, agree_x_id=AGREE_X_ID, agree_y_id=AGREE_Y_ID, bs=256, device=DEVICE):
    """Restricted-vocab (2-way) accuracy at t_agree positions, split by TRUE
    class -- not just aggregate. This split is the primary diagnostic in this
    notebook, not aggregate accuracy: each language's own role distribution is
    skewed ~90/10 by construction, so a model that ignores t2 entirely and
    always predicts the MAJORITY role for whatever BOS tag it sees already
    scores ~90% in aggregate, for every condition (shared or dedicated) --
    the exact Experiment 3 number-agreement trap ('aggregate accuracy
    conflating the easy 92% and hard 8% case') reappearing here. acc_majority
    / acc_minority report accuracy restricted to positions where the TRUE
    label is this language's own majority / minority role; majority_baseline
    is what a t2-blind majority-vote shortcut would score in aggregate, for
    comparison."""
    model.eval()
    n_x = n_y = correct_x = correct_y = 0
    for i in range(0, data.shape[0], bs):
        b = data[i:i + bs].to(device)
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            logits, _ = model(x)
        mask_x = (y == agree_x_id)
        mask_y = (y == agree_y_id)
        if mask_x.sum() + mask_y.sum() == 0:
            continue
        restricted = logits[..., [agree_x_id, agree_y_id]]
        pred = restricted.argmax(-1)          # 0 = X, 1 = Y
        correct_x += ((pred == 0) & mask_x).sum().item()
        correct_y += ((pred == 1) & mask_y).sum().item()
        n_x += mask_x.sum().item()
        n_y += mask_y.sum().item()
    acc_x = correct_x / max(1, n_x)
    acc_y = correct_y / max(1, n_y)
    n_total = n_x + n_y
    majority_is_y = n_y >= n_x
    majority_baseline = max(n_x, n_y) / max(1, n_total)
    acc_majority = acc_y if majority_is_y else acc_x
    acc_minority = acc_x if majority_is_y else acc_y
    n_minority = n_x if majority_is_y else n_y
    return dict(acc_X=acc_x, acc_Y=acc_y, n_X=n_x, n_Y=n_y,
                acc_majority=acc_majority, acc_minority=acc_minority,
                n_minority=n_minority, majority_baseline=majority_baseline)


@torch.no_grad()
def t_agree_accuracy(model, data, agree_x_id=AGREE_X_ID, agree_y_id=AGREE_Y_ID, bs=256, device=DEVICE):
    """Aggregate restricted-vocab accuracy -- kept for reference/back-compat
    ONLY. Do not use this as the primary metric for this design (see
    t_agree_accuracy_split's docstring); every results table below reports
    the split version, with aggregate shown as a secondary column at most."""
    r = t_agree_accuracy_split(model, data, agree_x_id, agree_y_id, bs, device)
    total = r["n_X"] + r["n_Y"]
    correct = r["acc_X"] * r["n_X"] + r["acc_Y"] * r["n_Y"]
    return correct / max(1, total), total


def silent(fn, *a, **k):
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)


def agg(v):
    import numpy as np
    v = np.array(v, float)
    return f"{np.nanmean(v):+.3f}\u00b1{np.nanstd(v):.3f}" if np.isfinite(v).any() else "  nan  "


## 10. Cheap gate (NO sweep yet): does an ISOLATED model actually learn this?

Requested check, run before any multi-seed/multi-d compute commitment: train ONE
small, cheap, single-seed dedicated model per language (`L_B`, `L_C`, and `L_A` as
a control) on nothing but that language's own data, and see whether it reaches
near-ceiling on the **minority-class** role &mdash; the specific thing the
mechanism is supposed to make learnable when there's no sharing pressure at all.

This is the direct analogue of two established gates in this project: Exp 3's
"isolated-anchor-only" diagnostic (run before spending the override-table sweep
budget) and Exp 6b's "terminal-recovery probe on finished checkpoints as a fast,
compute-free check before continuing to spend budget-matched sweep compute."

**Why this specific check, and not just aggregate `t_agree` accuracy:** each
language's OWN role distribution is skewed ~90/10 (that's the whole mechanism).
A model that ignores `t2` entirely and always predicts the MAJORITY role for
whichever BOS tag it sees will already score ~90% on aggregate `t_agree`
accuracy &mdash; for every condition, shared or dedicated. This is exactly the
Experiment 3 number-agreement trap ("model found the shortcut ignore the
quantifier, copy the noun's default... aggregate accuracy conflating the easy
92% and hard 8% case") reappearing here in a new grammar. **Aggregate accuracy
is not a valid primary metric for this design; minority-class accuracy is.**
`t_agree_accuracy` below (Section 10) is written to always report both, and
every results table downstream reports minority-class accuracy as the headline
number, aggregate as a secondary column.

If even an isolated, dedicated, single-language model can't beat the majority
baseline on its own minority class **after reweighting the loss** (Step 2's
mechanism, tested here too since a plain, unweighted run is expected to fail
on the rare class per the Exp 3 lesson below), the mechanism itself is broken
(not learnable at all, regardless of sharing) and nothing past this cell should
be trusted until that's fixed -- this is exactly the kind of thing that should
be caught HERE, not after the full multi-seed d-sweep + budget-matched
dedicated jobs + A' calibration have all been run.

In [12]:
GATE_D = 96          # generous capacity -- this gate asks "is it learnable at all", not "how cheaply"
GATE_STEPS = 30_000   # cheap: far below FIXED_BUDGET_SWEEP, single seed, single d, one job per language
GATE_SEED = 0
GATE_REWEIGHTS = [1, 20, 100]   # plain AND reweighted -- an unweighted near-zero minority accuracy is
                           # EXPECTED given the ~90/10 skew (the Exp 3 rare-event lesson), not by
                           # itself evidence the mechanism is broken. Only a minority accuracy that
                           # stays near-chance even WITH reweighting, in complete isolation with no
                           # sharing at all, means the mechanism itself needs fixing.
CK_GATE = "./exp7_gate"; os.makedirs(CK_GATE, exist_ok=True)

def gate_data_path(lang):
    return os.path.join(CK_GATE, f"data_gate_{lang}.pt")

def gate_ckpt_path(lang, w):
    tag = "baseline" if w == 1 else f"rw{w}"
    return os.path.join(CK_GATE, f"gate_{lang}_{tag}_d{GATE_D}.pt")

def minority_agree_id(lang):
    """Each language's OWN minority role, not a global constant: L_A's
    minority is ROLE-Y (AGREE_Y_ID); L_B/L_C's minority is ROLE-X
    (AGREE_X_ID), since they're built to favor Y ~90% of the time. This gate
    trains fully ISOLATED per-language models (unlike Step 2/3's shared model
    on the pooled mixture, where Y really is the aggregate-rarer class
    overall and AGREE_Y_ID is the right constant everywhere) -- reweighting
    the wrong class here would upweight a language's MAJORITY instead of its
    minority, making minority accuracy worse as w grows, not better."""
    return AGREE_Y_ID if lang == "L_A" else AGREE_X_ID

gate_jobs = []
for lang in LANG_NAMES:
    dp = gate_data_path(lang)
    if not os.path.exists(dp):
        data = make_dual_role_monolingual(rules, lang, n_trees=20_000, seed=1, seq_len=SEQ_LEN_CHECK)
        torch.save(data.cpu(), dp)
    for w in GATE_REWEIGHTS:
        gate_jobs.append(dict(d=GATE_D, seed=GATE_SEED, n_head=N_HEAD, steps=GATE_STEPS,
                              data_path=dp, ckpt_path=gate_ckpt_path(lang, w),
                              reweight=float(w), agree_y_id=minority_agree_id(lang)))

run_parallel_jobs(gate_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=6)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in gate_jobs])

print(f"\n{'lang':>4s} {'w':>3s} {'majority_baseline':>18s} {'acc_majority':>13s} {'acc_MINORITY':>13s} {'n_minority':>10s}")
gate_results = {}
for lang in LANG_NAMES:
    for w in GATE_REWEIGHTS:
        m = load_shared_ckpt(gate_ckpt_path(lang, w), GATE_D, SEQ_LEN_CHECK)
        fresh = make_dual_role_monolingual(rules, lang, n_trees=3000, seed=2222, seq_len=SEQ_LEN_CHECK)
        r = t_agree_accuracy_split(m, fresh)
        gate_results[(lang, w)] = r
        print(f"{lang:>4s} {w:>3d} {r['majority_baseline']:>18.3f} {r['acc_majority']:>13.3f} "
              f"{r['acc_minority']:>13.3f} {r['n_minority']:>10d}")
        del m; torch.cuda.empty_cache()

print("\nSTOP AND CHECK before proceeding to Section 11's full sweep:")
print("  - w=1, acc_MINORITY near-zero is EXPECTED (rare-event training signal,")
print("    matches Exp 3) -- not itself a red flag.")
print("  - w=20, acc_MINORITY should climb well above the w=1 number and well")
print("    above chance. If it does NOT -- if even a reweighted, fully isolated,")
print("    single-language model can't learn its own minority class -- the")
print("    mechanism itself needs fixing (grammar/weights/steps/window), before")
print("    any of Section 11+'s multi-seed, multi-d, budget-matched compute is")
print("    worth spending.")


9 runs to do, 6 at a time
finished d=96 seed=0 (gate_L_A_baseline_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_B_baseline_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_A_rw100_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_A_rw20_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_B_rw20_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_B_rw100_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_C_baseline_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_C_rw20_d96.pt)  exit code 0
finished d=96 seed=0 (gate_L_C_rw100_d96.pt)  exit code 0
all queued jobs finished (check exit codes above for failures)
all 9 runs complete

lang   w  majority_baseline  acc_majority  acc_MINORITY n_minority
./exp7_gate/gate_L_A_baseline_d96.pt: step=30000, tail-1000 mean loss=0.6655
 L_A   1              0.891         0.957         0.040        324
./exp7_gate/gate_L_A_rw20_d96.pt: step=30000, tail-1000 mean loss=0.7784
 L_A  20              0.891         0.881         0.102        324
./exp7_gate

In [18]:
CK7 = "./exp7_gate"

## 11. Training &mdash; Step 1: baseline shared vs. budget-matched dedicated

(a) shared d-sweep on the full mixture, (b) budget-matched dedicated models for
B/C individually (data-volume-matched per Section 5), (c) A' calibration control.

In [14]:
def shared_path(d, seed, tag="baseline"):
    return os.path.join(CK7, f"shared_{tag}_d{d}_seed{seed}.pt")

def dedic_path(lang, dprime, seed, tag="baseline"):
    return os.path.join(CK7, f"dedic_{tag}_{lang}_d{dprime}_seed{seed}.pt")

def shared_data_path(seed, tag="baseline"):
    return os.path.join(CK7, f"data_shared_{tag}_seed{seed}.pt")

def dedic_data_path(lang, seed, target_chunks, tag="baseline"):
    return os.path.join(CK7, f"data_dedic_{tag}_{lang}_seed{seed}.pt")

In [ ]:
CK7 = "./exp7_dualrole"; os.makedirs(CK7, exist_ok=True)

def shared_path(d, seed, tag="baseline"):
    return os.path.join(CK7, f"shared_{tag}_d{d}_seed{seed}.pt")

def dedic_path(lang, dprime, seed, tag="baseline"):
    return os.path.join(CK7, f"dedic_{tag}_{lang}_d{dprime}_seed{seed}.pt")

def shared_data_path(seed, tag="baseline"):
    return os.path.join(CK7, f"data_shared_{tag}_seed{seed}.pt")

def dedic_data_path(lang, seed, target_chunks, tag="baseline"):
    return os.path.join(CK7, f"data_dedic_{tag}_{lang}_seed{seed}.pt")

# --- 1) precompute datasets (shared + data-volume-matched dedicated) ---
for seed in SEEDS3:
    data_seed = seed if seed != 0 else 1
    p = shared_data_path(data_seed)
    if not os.path.exists(p):
        data = make_dual_role_mix(rules, n_trees=N_TREES_CHECK, seed=data_seed, seq_len=SEQ_LEN_CHECK, mix=MIX)
        torch.save(data.cpu(), p)
        print(f"saved {p}  ({data.shape[0]} chunks)")

_shared_chunks = torch.load(shared_data_path(SEEDS3[1] if SEEDS3[0] == 0 else SEEDS3[0]),
                            map_location="cpu", weights_only=False).shape[0]

for lang in ("L_B", "L_C"):
    for seed in SEEDS3:
        data_seed = seed if seed != 0 else 1
        p = dedic_data_path(lang, data_seed, _shared_chunks)
        if not os.path.exists(p):
            data = make_dual_role_dedicated_matched(rules, lang, data_seed, SEQ_LEN_CHECK,
                                                     target_chunks=_shared_chunks, mix=MIX)
            torch.save(data.cpu(), p)
            print(f"saved {p}  ({data.shape[0]} chunks)")

# --- 2) launch shared d-sweep ---
step1_shared_jobs = []
for d in D_LIST:
    for seed in SEEDS3:
        data_seed = seed if seed != 0 else 1
        step1_shared_jobs.append(dict(d=d, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                      data_path=shared_data_path(data_seed),
                                      ckpt_path=shared_path(d, seed)))
run_parallel_jobs(step1_shared_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in step1_shared_jobs])

# --- 3) launch budget-matched dedicated baselines ---
step1_dedic_jobs = []
seen_dprime = set()
for d in D_LIST:
    dprime = BM_WIDTHS[d]
    if dprime in seen_dprime:
        continue
    seen_dprime.add(dprime)
    for lang in ("L_B", "L_C"):
        for seed in SEEDS3:
            data_seed = seed if seed != 0 else 1
            step1_dedic_jobs.append(dict(d=dprime, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                         data_path=dedic_data_path(lang, data_seed, _shared_chunks),
                                         ckpt_path=dedic_path(lang, dprime, seed)))
run_parallel_jobs(step1_dedic_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in step1_dedic_jobs])


### A' calibration control (reused pattern from Exp 6b)

A second anchor-identical language (same linearization as L_A, different BOS
tag), mixed in at the anchor's expense, as the "should show ~0 gap" sanity
ceiling for any sharing instrument.

In [ ]:
LANGS_A = dict(LANGS); LANGS_A["L_A_PRIME"] = set()
LANG_NAMES_A = LANG_NAMES + ["L_A_PRIME"]
MIX_A = {"L_A": 0.35, "L_A_PRIME": 0.35, "L_B": 0.15, "L_C": 0.15}
BOS_A = {name: TERM_SIZE + i for i, name in enumerate(LANG_NAMES_A)}
VOCAB_SIZE_A_BASE = TERM_SIZE + len(LANG_NAMES_A)
AGREE_X_ID_A = VOCAB_SIZE_A_BASE
AGREE_Y_ID_A = VOCAB_SIZE_A_BASE + 1
VOCAB_SIZE_A = VOCAB_SIZE_A_BASE + 2
RULE_WEIGHTS_A = dict(RULE_WEIGHTS); RULE_WEIGHTS_A["L_A_PRIME"] = RULE_WEIGHTS["L_A"]

def make_dual_role_mix_aprime(rules, n_trees, seed, seq_len, mix=MIX_A):
    rng = random.Random(seed)
    stream = []
    for _ in range(n_trees):
        lang = rng.choices(LANG_NAMES_A, weights=[mix[n] for n in LANG_NAMES_A])[0]
        tree, role = sample_dual_role_tree(rules, RULE_WEIGHTS_A[lang], NT_X, NT_Y, rng)
        toks, _ = linearize_marked(tree, LANGS_A[lang])
        stream.append(BOS_A[lang])
        stream.extend(toks)
        stream.append(AGREE_X_ID_A if role == "X" else AGREE_Y_ID_A)
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)


APRIME_D_LIST = [16, 32, 96]
CK_APRIME = "./exp7_aprime"; os.makedirs(CK_APRIME, exist_ok=True)

def aprime_path(d, seed):
    return os.path.join(CK_APRIME, f"shared_aprime_d{d}_seed{seed}.pt")

def aprime_data_path(seed):
    return os.path.join(CK_APRIME, f"data_aprime_seed{seed}.pt")

for seed in SEEDS3:
    data_seed = seed if seed != 0 else 1
    p = aprime_data_path(data_seed)
    if not os.path.exists(p):
        data = make_dual_role_mix_aprime(rules, n_trees=N_TREES_CHECK, seed=data_seed, seq_len=SEQ_LEN_CHECK)
        torch.save(data.cpu(), p)
        print(f"saved {p}")

aprime_jobs = []
for d in APRIME_D_LIST:
    for seed in SEEDS3:
        data_seed = seed if seed != 0 else 1
        aprime_jobs.append(dict(d=d, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                data_path=aprime_data_path(data_seed), ckpt_path=aprime_path(d, seed)))
run_parallel_jobs(aprime_jobs, vocab_size=VOCAB_SIZE_A, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in aprime_jobs])


## 12. Step 1 results: `t_agree` accuracy, shared vs. dedicated

"This step alone establishes whether there's a gap at all &mdash; expect there to
be one, based on the mechanism's design." A large fresh eval set is drawn per
language/condition (not the training data) for an unbiased read.

In [ ]:
N_EVAL_TREES = 5000
EVAL_SEED = 98765

def eval_shared_t_agree(d, seed, n_eval=N_EVAL_TREES, eval_seed=EVAL_SEED):
    m = silent(load_shared_ckpt, shared_path(d, seed), d, SEQ_LEN_CHECK)
    fresh = make_dual_role_mix(rules, n_trees=n_eval, seed=eval_seed, seq_len=SEQ_LEN_CHECK, mix=MIX)
    out = {}
    for lang in LANG_NAMES:
        rows = fresh[fresh[:, 0] == BOS[lang]]
        out[lang] = t_agree_accuracy_split(m, rows) if rows.shape[0] > 0 else None
    del m; torch.cuda.empty_cache()
    return out


def eval_dedicated_t_agree(lang, dprime, seed, n_eval=3000, eval_seed=EVAL_SEED):
    m = silent(load_shared_ckpt, dedic_path(lang, dprime, seed), dprime, SEQ_LEN_CHECK)
    fresh = make_dual_role_monolingual(rules, lang, n_eval, eval_seed, SEQ_LEN_CHECK)
    r = t_agree_accuracy_split(m, fresh)
    del m; torch.cuda.empty_cache()
    return r


# headline metric is MINORITY-class accuracy (see Section 9/10) -- aggregate
# would sit near each language's own ~90% majority-baseline regardless of
# whether the model is actually reading t2's role or just its BOS tag.
step1_results = {}
print(f"{'d':>4s} {'lang':>4s} {'minacc_shared':>13s} {'minacc_dedic':>13s} {'gap':>8s} "
      f"{'d_prime':>7s} {'maj_baseline':>12s}")
for d in D_LIST:
    dprime = BM_WIDTHS[d]
    shared_r = eval_shared_t_agree(d, seed=SEEDS3[0])
    for lang in ("L_B", "L_C"):
        dedic_r = eval_dedicated_t_agree(lang, dprime, seed=SEEDS3[0])
        gap = shared_r[lang]["acc_minority"] - dedic_r["acc_minority"]
        step1_results[(d, lang)] = dict(shared=shared_r[lang], dedicated=dedic_r, gap=gap)
        print(f"{d:>4d} {lang:>4s} {shared_r[lang]['acc_minority']:>13.3f} "
              f"{dedic_r['acc_minority']:>13.3f} {gap:>+8.3f} {dprime:>7d} "
              f"{dedic_r['majority_baseline']:>12.3f}")

print("\n(negative gap = dedicated beats shared on the MINORITY-class read at t_agree")
print(" -- the candidate ε signature. Compare each acc_minority to maj_baseline: if")
print(" acc_minority sits at or below maj_baseline's complement, the model may just")
print(" be guessing majority everywhere and getting minority positions wrong by")
print(" construction -- that alone does not distinguish real ε from Step 2's artifact.)")
print("Do not trust this gap yet -- proceed to Step 2 (reweighted-loss control) first.")


## 13. Step 2: reweighted-loss control

Retrain the **shared** model only, with per-token loss upweighted at
`AGREE_Y_ID` positions (the case ROLE-Y-dependent B/C need but the anchor-
dominated mixture rarely reinforces). Dedicated baselines are NOT reweighted
(for them ROLE-Y isn't rare, so there's nothing to correct). If a high-enough
`w` closes the Step 1 gap, it was an optimization-incentive artifact (mirrors
the Exp 3 number-agreement finding), not genuine &epsilon;.

In [ ]:
REWEIGHT_D_LIST = [16, 96]   # representative capacities first, per the design doc

step2_jobs = []
for d in REWEIGHT_D_LIST:
    for w in REWEIGHT_FACTORS:
        if w == 1:
            continue   # w=1 is just the Step 1 shared run, already have it
        for seed in SEEDS3:
            data_seed = seed if seed != 0 else 1
            step2_jobs.append(dict(d=d, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                   data_path=shared_data_path(data_seed),
                                   ckpt_path=shared_path(d, seed, tag=f"rw{w}"),
                                   reweight=float(w), agree_y_id=AGREE_Y_ID))
run_parallel_jobs(step2_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in step2_jobs])

step2_results = {}
print(f"{'d':>4s} {'w':>4s} {'lang':>4s} {'minacc_shared_rw':>16s} {'minacc_dedic':>13s} {'gap':>8s}")
for d in REWEIGHT_D_LIST:
    dprime = BM_WIDTHS[d]
    for w in REWEIGHT_FACTORS:
        tag = "baseline" if w == 1 else f"rw{w}"
        shared_r = {}
        for seed in SEEDS3:
            m = silent(load_shared_ckpt, shared_path(d, seed, tag=tag), d, SEQ_LEN_CHECK)
            fresh = make_dual_role_mix(rules, n_trees=N_EVAL_TREES, seed=EVAL_SEED, seq_len=SEQ_LEN_CHECK, mix=MIX)
            for lang in ("L_B", "L_C"):
                rows = fresh[fresh[:, 0] == BOS[lang]]
                r = t_agree_accuracy_split(m, rows)
                shared_r.setdefault(lang, []).append(r["acc_minority"])
            del m; torch.cuda.empty_cache()
        for lang in ("L_B", "L_C"):
            acc_shared = sum(shared_r[lang]) / len(shared_r[lang])
            acc_dedic = step1_results[(d, lang)]["dedicated"]["acc_minority"]
            gap = acc_shared - acc_dedic
            step2_results[(d, w, lang)] = dict(acc_shared=acc_shared, acc_dedic=acc_dedic, gap=gap)
            print(f"{d:>4d} {w:>4d} {lang:>4s} {acc_shared:>16.3f} {acc_dedic:>13.3f} {gap:>+8.3f}")

print("\nDecision rule: if the gap closes (-> ~0) at the largest w tested, Step 1's")
print("gap was an optimization-incentive artifact -- report that, do not claim ε>0.")
print("If the gap survives even at the largest w, proceed to Step 3 as a real candidate.")


## 14. Step 3: N-sweep at fixed &rho;

Uses whichever loss-weighting survived Step 2 (or plain loss if reweighting made
no difference) across the full `d`-grid, convergence-matched (fixed step budget,
matched across shared/dedicated as established in Exp 6). Plots the gap as a
function of `d` to distinguish finite-capacity slack (gap &rarr; 0) from a
persistent floor (gap plateaus).

In [16]:
SURVIVING_REWEIGHT = 1   # <-- update to the smallest w that closed/stabilized the gap in Step 2, or 1 if none did

step3_tag = "baseline" if SURVIVING_REWEIGHT == 1 else f"rw{SURVIVING_REWEIGHT}"

In [ ]:
# set this after inspecting Step 2's printout above
SURVIVING_REWEIGHT = 1   # <-- update to the smallest w that closed/stabilized the gap in Step 2, or 1 if none did

step3_tag = "baseline" if SURVIVING_REWEIGHT == 1 else f"rw{SURVIVING_REWEIGHT}"

if SURVIVING_REWEIGHT != 1:
    step3_jobs = []
    for d in D_LIST:
        if d in REWEIGHT_D_LIST:
            continue   # already trained in Step 2
        for seed in SEEDS3:
            data_seed = seed if seed != 0 else 1
            step3_jobs.append(dict(d=d, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                                   data_path=shared_data_path(data_seed),
                                   ckpt_path=shared_path(d, seed, tag=step3_tag),
                                   reweight=float(SURVIVING_REWEIGHT), agree_y_id=AGREE_Y_ID))
    run_parallel_jobs(step3_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
    assert_all_complete([(j["ckpt_path"], j["steps"]) for j in step3_jobs])

step3_results = {}
print(f"{'d':>4s} {'lang':>4s} {'minacc_shared':>13s} {'minacc_dedic':>13s} {'gap':>8s}")
for d in D_LIST:
    dprime = BM_WIDTHS[d]
    shared_r = {lang: [] for lang in ("L_B", "L_C")}
    for seed in SEEDS3:
        m = silent(load_shared_ckpt, shared_path(d, seed, tag=step3_tag), d, SEQ_LEN_CHECK)
        fresh = make_dual_role_mix(rules, n_trees=N_EVAL_TREES, seed=EVAL_SEED, seq_len=SEQ_LEN_CHECK, mix=MIX)
        for lang in ("L_B", "L_C"):
            rows = fresh[fresh[:, 0] == BOS[lang]]
            r = t_agree_accuracy_split(m, rows)
            shared_r[lang].append(r["acc_minority"])
        del m; torch.cuda.empty_cache()
    for lang in ("L_B", "L_C"):
        acc_shared = sum(shared_r[lang]) / len(shared_r[lang])
        acc_dedic = step1_results[(d, lang)]["dedicated"]["acc_minority"]   # dedicated unaffected by reweighting
        gap = acc_shared - acc_dedic
        step3_results[(d, lang)] = dict(acc_shared=acc_shared, acc_dedic=acc_dedic, gap=gap)
        print(f"{d:>4d} {lang:>4s} {acc_shared:>13.3f} {acc_dedic:>13.3f} {gap:>+8.3f}")

print("\nGap shrinking toward zero as d grows -> finite-capacity slack, not persistent ε.")
print("Gap plateauing at a non-zero floor -> current best candidate for real ε (extend")
print("the d-grid further before concluding this, per the Exp 6b d=96 word-order lesson).")


## 15. Step 4 (secondary): &rho;-sweep at fixed d

Vary the anchor's mixture share. If the persistent gap grows with &rho; and
shrinks toward 0 as &rho;&rarr;1/K (balanced mixture), that confirms the
training-skew story specifically. **Note:** per the design doc, this changes
&epsilon; into a function of &rho; &mdash; a change to the frozen derivation
document. Do not fold this back into `cost_derivation.md` until the result
actually supports it.

In [ ]:
RHO_D = D_LIST[0]   # the d that showed the clearest Step 3 signal -- update after inspecting Step 3

def mix_at_rho(rho):
    rest = (1 - rho) / 2
    return {"L_A": rho, "L_B": rest, "L_C": rest}

step4_jobs = []
for rho in RHO_SWEEP:
    mix_r = mix_at_rho(rho)
    for seed in SEEDS3:
        data_seed = seed if seed != 0 else 1
        dp = os.path.join(CK7, f"data_shared_rho{rho}_seed{data_seed}.pt")
        if not os.path.exists(dp):
            data = make_dual_role_mix(rules, n_trees=N_TREES_CHECK, seed=data_seed, seq_len=SEQ_LEN_CHECK, mix=mix_r)
            torch.save(data.cpu(), dp)
        step4_jobs.append(dict(d=RHO_D, seed=seed, n_head=N_HEAD, steps=FIXED_BUDGET_SWEEP,
                               data_path=dp, ckpt_path=shared_path(RHO_D, seed, tag=f"rho{rho}"),
                               reweight=float(SURVIVING_REWEIGHT), agree_y_id=AGREE_Y_ID))
run_parallel_jobs(step4_jobs, vocab_size=VOCAB_SIZE, seq_len=SEQ_LEN_CHECK, max_par=MAX_PAR)
assert_all_complete([(j["ckpt_path"], j["steps"]) for j in step4_jobs])

step4_results = {}
print(f"{'rho':>5s} {'lang':>4s} {'acc_shared':>10s}")
for rho in RHO_SWEEP:
    mix_r = mix_at_rho(rho)
    shared_r = {lang: [] for lang in ("L_B", "L_C")}
    for seed in SEEDS3:
        m = silent(load_shared_ckpt, shared_path(RHO_D, seed, tag=f"rho{rho}"), RHO_D, SEQ_LEN_CHECK)
        fresh = make_dual_role_mix(rules, n_trees=N_EVAL_TREES, seed=EVAL_SEED, seq_len=SEQ_LEN_CHECK, mix=mix_r)
        for lang in ("L_B", "L_C"):
            rows = fresh[fresh[:, 0] == BOS[lang]]
            acc, n = t_agree_accuracy(m, rows)
            shared_r[lang].append(acc)
        del m; torch.cuda.empty_cache()
    for lang in ("L_B", "L_C"):
        acc_shared = sum(shared_r[lang]) / len(shared_r[lang])
        step4_results[(rho, lang)] = acc_shared
        print(f"{rho:>5.2f} {lang:>4s} {acc_shared:>10.3f}")

print("\nExpect acc_shared to fall (gap to widen) as rho rises toward 1, and rise back")
print("toward the dedicated ceiling as rho -> 1/3 (balanced mixture).")


## 16. Step 5 (only if Step 3 shows a real, persistent gap): representational probe

Two instruments, reused from Exp 6b's pattern: (a) a direct linear probe on
`t2`'s own hidden state predicting its role, comparing shared vs. dedicated
models; (b) interchange intervention (patch `t2`'s activation from a
dedicated, ROLE-Y-native model into the shared model's forward pass, check
whether this fixes the `t_agree` prediction). A clean Step 3 CE/accuracy gap
alone is NOT proof of a shared subspace (the Bridge-1 conflation already
flagged in project memory) &mdash; this section is the representational-level
confirmation, not a restatement of Step 3.

In [19]:
def t2_hidden_states(model, probe_set, lang, layer):
    H = extract_hidden_states(model, probe_set[lang]["idx"], [layer])[layer]
    idx = torch.arange(H.shape[0])
    return H[idx, probe_set[lang]["t2_pos"]]   # (n_trees, d_model) -- t2's own hidden state


def _fit_logreg(X, y, n_class, epochs=300, lr=1e-2, device=DEVICE):
    mu, sd = X.mean(0), X.std(0) + 1e-6
    Xs = ((X - mu) / sd).to(device); ys = y.to(device)
    W = nn.Linear(X.shape[1], n_class).to(device)
    opt = torch.optim.AdamW(W.parameters(), lr=lr)
    for _ in range(epochs):
        loss = F.cross_entropy(W(Xs), ys); opt.zero_grad(); loss.backward(); opt.step()
    return W, mu, sd


@torch.no_grad()
def _score(W, mu, sd, X, y, device=DEVICE):
    return (W(((X - mu) / sd).to(device)).argmax(-1).cpu() == y).float().mean().item()


def role_probe(model, probe_set, lang, layer, train_ids, test_ids):
    X = t2_hidden_states(model, probe_set, lang, layer)
    y = probe_set[lang]["role"]
    train_mask = torch.zeros(X.shape[0], dtype=torch.bool); train_mask[list(train_ids)] = True
    test_mask = ~train_mask
    W, mu, sd = _fit_logreg(X[train_mask], y[train_mask], n_class=2)
    return _score(W, mu, sd, X[test_mask], y[test_mask])


PROBE_D = D_LIST[-1]   # start at the largest-capacity point; extend to the full grid as needed
N_PROBE_TREES = 500
probe_set = build_matched_probe_set(rules, N_PROBE_TREES, seed=555, seq_len=SEQ_LEN_CHECK)
train_ids, test_ids = split_trees(probe_set["L_A"]["idx"].shape[0], test_frac=0.2, seed=0)

print(f"{'d':>4s} {'lang':>4s} {'layer':>5s} {'shared_acc':>10s} {'dedic_acc':>10s}")
role_probe_results = {}
for layer in (8, 11):
    m_shared = silent(load_shared_ckpt, shared_path(PROBE_D, SEEDS3[0], tag=step3_tag), PROBE_D, SEQ_LEN_CHECK)
    for lang in ("L_B", "L_C"):
        acc_shared = role_probe(m_shared, probe_set, lang, layer, train_ids, test_ids)
        dprime = BM_WIDTHS[PROBE_D]
        m_dedic = silent(load_shared_ckpt, dedic_path(lang, dprime, SEEDS3[0]), dprime, SEQ_LEN_CHECK)
        acc_dedic = role_probe(m_dedic, probe_set, lang, layer, train_ids, test_ids)
        role_probe_results[(PROBE_D, lang, layer)] = dict(shared=acc_shared, dedicated=acc_dedic)
        print(f"{PROBE_D:>4d} {lang:>4s} {layer:>5d} {acc_shared:>10.3f} {acc_dedic:>10.3f}")
        del m_dedic; torch.cuda.empty_cache()
    del m_shared; torch.cuda.empty_cache()

print("\nA clean shared-vs-dedicated role-probe gap here, alongside Step 3's behavioral")
print("gap, is the representational [R] confirmation. If Step 3 shows a gap but this")
print("probe does not, investigate before reporting either direction (per the Step 6")
print("'instrument failure' criterion in the design doc).")


   d lang layer shared_acc  dedic_acc


FileNotFoundError: [Errno 2] No such file or directory: './exp7_gate/shared_baseline_d96_seed0.pt'

## 17. Wrap-up

Decision tree from the design doc, restated for quick reference once the cells
above have been run:

- **Clean success:** Step 2 gap survives reweighting, Step 3 gap plateaus and
  doesn't close by d=96 (extend the grid further to check), Step 4 shows the
  &rho;-dependence signature, Step 5 confirms it representationally.
- **Partial/negative, still worth reporting:** gap closes at Step 2 (incentive
  artifact) or Step 3 (finite-capacity slack) &mdash; a rigorous elimination,
  not a failure.
- **Instrument failure:** any Section 3 pre-flight check fails, or Step 5's
  probe contradicts Step 3's behavioral result &mdash; investigate before
  reporting either direction.

**Open items flagged for review, not silently resolved above:**
1. The `t_agree`-as-postamble fix (Section 0 markdown) is a deviation from a
   literal reading of the design doc and should be confirmed before trusting
   any L_C result.
2. The position-controlled trunk (Section 2: fixed 3-way root split, single
   path down to `ROLE_CHOICE`) replaces the original design's "let the
   ordinary grammar decide where NT_X/NT_Y sit" approach, after empirically
   finding that approach put `t2` a median 7 tokens from the readout for
   `L_C` (vs. ~240 for `L_A`/`L_B`) &mdash; effectively no long-distance
   dependency at all for `L_C`. This is a substantive structural change from
   the handoff design doc and should be reviewed, not just the accuracy
   numbers it produces.
3. `SURVIVING_REWEIGHT` (Section 14) and `RHO_D` (Section 15) need to be set
   by hand after inspecting Step 2/3's printed results; they are not
   auto-selected.


In [21]:
def gate_ckpt_path(lang, w):
    if w == 1:
        tag = "baseline"
    elif w == 20:
        tag = "rw20"
    elif w == 100:
        tag = "rw100"
    else:
        raise ValueError(f"Unsupported w={w}")

    return f"./exp7_gate/gate_{lang}_{tag}_d96.pt"

In [22]:
PROBE_D = D_LIST[-1]
N_PROBE_TREES = 500

probe_set = build_matched_probe_set(
    rules,
    N_PROBE_TREES,
    seed=555,
    seq_len=SEQ_LEN_CHECK
)

train_ids, test_ids = split_trees(
    probe_set["L_A"]["idx"].shape[0],
    test_frac=0.2,
    seed=0
)

print(f"{'d':>4s} {'lang':>4s} {'layer':>5s} {'shared_acc':>10s} {'dedic_acc':>10s}")

role_probe_results = {}

# Choose which shared checkpoint you want here
SHARED_LANG = "L_A"
SHARED_W = 20

m_shared = silent(
    load_shared_ckpt,
    gate_ckpt_path(SHARED_LANG, SHARED_W),
    PROBE_D,
    SEQ_LEN_CHECK
)

for layer in (8, 11):

    for lang in ("L_B", "L_C"):

        # Shared model
        acc_shared = role_probe(
            m_shared,
            probe_set,
            lang,
            layer,
            train_ids,
            test_ids
        )

        # Dedicated model for this language
        dprime = BM_WIDTHS[PROBE_D]

        m_dedic = silent(
            load_shared_ckpt,
            dedic_path(lang, dprime, SEEDS3[0]),
            dprime,
            SEQ_LEN_CHECK
        )

        acc_dedic = role_probe(
            m_dedic,
            probe_set,
            lang,
            layer,
            train_ids,
            test_ids
        )

        role_probe_results[(PROBE_D, lang, layer)] = {
            "shared": acc_shared,
            "dedicated": acc_dedic
        }

        print(
            f"{PROBE_D:>4d} {lang:>4s} {layer:>5d} "
            f"{acc_shared:>10.3f} {acc_dedic:>10.3f}"
        )

        del m_dedic
        torch.cuda.empty_cache()

del m_shared
torch.cuda.empty_cache()

print(
    "\nA clean shared-vs-dedicated role-probe gap here, alongside Step 3's behavioral"
)
print(
    "gap, is the representational [R] confirmation. If Step 3 shows a gap but this"
)
print(
    "probe does not, investigate before reporting either direction (per the Step 6"
)
print(
    "'instrument failure' criterion in the design doc)."
)

   d lang layer shared_acc  dedic_acc


FileNotFoundError: [Errno 2] No such file or directory: './exp7_gate/dedic_baseline_L_B_d56_seed0.pt'

In [23]:
PROBE_D = D_LIST[-1]
N_PROBE_TREES = 500

probe_set = build_matched_probe_set(
    rules,
    N_PROBE_TREES,
    seed=555,
    seq_len=SEQ_LEN_CHECK
)

train_ids, test_ids = split_trees(
    probe_set["L_A"]["idx"].shape[0],
    test_frac=0.2,
    seed=0
)

print(f"{'d':>4s} {'lang':>4s} {'layer':>5s} {'shared_acc':>10s} {'dedic_acc':>10s}")

role_probe_results = {}

# L_A rw20 checkpoint is the shared model
m_shared = silent(
    load_shared_ckpt,
    "./exp7_gate/gate_L_A_rw20_d96.pt",
    PROBE_D,
    SEQ_LEN_CHECK
)

for layer in (8, 11):

    for lang in ("L_B", "L_C"):

        acc_shared = role_probe(
            m_shared,
            probe_set,
            lang,
            layer,
            train_ids,
            test_ids
        )

        # Dedicated baseline checkpoint for the language being probed
        m_dedic = silent(
            load_shared_ckpt,
            f"./exp7_gate/gate_{lang}_baseline_d96.pt",
            PROBE_D,
            SEQ_LEN_CHECK
        )

        acc_dedic = role_probe(
            m_dedic,
            probe_set,
            lang,
            layer,
            train_ids,
            test_ids
        )

        role_probe_results[(PROBE_D, lang, layer)] = {
            "shared": acc_shared,
            "dedicated": acc_dedic,
        }

        print(
            f"{PROBE_D:>4d} {lang:>4s} {layer:>5d} "
            f"{acc_shared:>10.3f} {acc_dedic:>10.3f}"
        )

        del m_dedic
        torch.cuda.empty_cache()

del m_shared
torch.cuda.empty_cache()

   d lang layer shared_acc  dedic_acc
  96  L_B     8      0.830      0.850
  96  L_C     8      0.800      0.760
  96  L_B    11      0.840      0.850
  96  L_C    11      0.810      0.790


In [25]:
PROBE_D = D_LIST[-1]
N_PROBE_TREES = 500

probe_set = build_matched_probe_set(
    rules,
    N_PROBE_TREES,
    seed=555,
    seq_len=SEQ_LEN_CHECK
)

train_ids, test_ids = split_trees(
    probe_set["L_A"]["idx"].shape[0],
    test_frac=0.2,
    seed=0
)

print(f"{'d':>4s} {'lang':>4s} {'layer':>5s} {'role_acc':>10s}")

role_probe_results = {}

# The ONLY model we have
m = silent(
    load_shared_ckpt,
    "./exp7_gate/gate_L_A_rw20_d96.pt",
    PROBE_D,
    SEQ_LEN_CHECK
)

for layer in (8, 11):

    for lang in ("L_A",):

        acc = role_probe(
            m,
            probe_set,
            lang,
            layer,
            train_ids,
            test_ids
        )

        role_probe_results[(PROBE_D, lang, layer)] = acc

        print(
            f"{PROBE_D:>4d} {lang:>4s} {layer:>5d} "
            f"{acc:>10.3f}"
        )

del m
torch.cuda.empty_cache()

   d lang layer   role_acc
  96  L_A     8      0.840
  96  L_A    11      0.820
